In [ ]:
# 将 llm_toy/src 加入 sys.path（兼容多启动位置）
import sys
from pathlib import Path

def _add_src_path():
    candidates = [
        Path.cwd() / 'llm_toy' / 'src',           # 在项目根启动Jupyter
        Path.cwd() / 'src',                        # 在 llm_toy 目录启动
        Path.cwd().parent / 'llm_toy' / 'src',
        Path.cwd().parent / 'src',
    ]
    # 向上回溯几层尝试
    for base in list(Path.cwd().parents)[:3]:
        candidates.append(base / 'llm_toy' / 'src')
        candidates.append(base / 'src')
    for p in candidates:
        if (p / 'model.py').exists() and (p / 'utils.py').exists():
            sys.path.append(str(p.resolve()))
            print('已添加src路径:', p.resolve())
            return str(p.resolve())
    print('警告：未找到 llm_toy/src，请手动添加路径或调整工作目录。')
    return None

SRC_PATH = _add_src_path()


# 简单 LLM 演示

本 Notebook 演示使用预训练模型的基本 LLM 功能。

In [ ]:
import sys
import os

from model import SimpleGPTModel
from utils import set_seed, get_device, print_gpu_memory

# 设置随机种子以保证可复现性
set_seed(42)

# 获取设备
device = get_device()
print_gpu_memory()

In [ ]:
# 初始化 GPT 模型
print("正在加载 GPT-2 模型...")
gpt_model = SimpleGPTModel(model_name="gpt2")

# 获取模型信息
model_info = gpt_model.get_model_info()
print("\n模型信息:")
for key, value in model_info.items():
    print(f"{key}: {value}")

In [ ]:
# 测试文本生成
prompts = [
    "The future of artificial intelligence is",
    "Machine learning can help us",
    "In the world of technology,",
    "The most important programming language for AI is"
]

print("文本生成示例:")
print("=" * 60)

for i, prompt in enumerate(prompts, 1):
    print(f"\n示例 {i}:")
    print(f"提示词: {prompt}")
    
    # 使用不同参数生成文本
    generated = gpt_model.generate_text(
        prompt,
        max_length=50,
        temperature=0.7,
        do_sample=True
    )
    
    print(f"生成结果: {generated}")
    print("-" * 60)

In [ ]:
# 测试不同生成参数
test_prompt = "Deep learning is"

print("测试不同生成参数:")
print("=" * 60)

# 低温度 (更确定性)
print("\n低温度 (0.3) - 更确定性:")
result1 = gpt_model.generate_text(test_prompt, max_length=30, temperature=0.3)
print(result1)

# 高温度 (更随机)
print("\n高温度 (1.2) - 更随机:")
result2 = gpt_model.generate_text(test_prompt, max_length=30, temperature=1.2)
print(result2)

# 贪婪解码 (temperature=0, do_sample=False)
print("\n贪婪解码 (Greedy Decoding):")
result3 = gpt_model.generate_text(test_prompt, max_length=30, temperature=1.0, do_sample=False)
print(result3)

## 使用在线大模型 (DeepSeek V3.2 Exp)

如果您配置了 SiliconFlow API Key，可以尝试使用更强大的 DeepSeek V3.2 Exp 模型。

In [ ]:
# 使用在线大模型 (DeepSeek V3.2 Exp)
# 请确保已在 configs/llm_api_config.json 中配置了 SILICONFLOW_API_KEY

try:
    from online_model import create_online_model
    
    print("正在初始化在线模型 (DeepSeek V3.2 Exp)...")
    # provider='siliconflow' 将使用我们在 configs 中配置的 DeepSeek 模型
    online_model = create_online_model(provider="siliconflow")
    
    prompt = "请解释一下量子纠缠是什么，用通俗易懂的语言。"
    print(f"\n提示词: {prompt}")
    
    response = online_model.generate_text(prompt, max_length=500)
    print(f"\n生成结果:\n{response}")
    
except Exception as e:
    print(f"无法初始化在线模型: {e}")
    print("请检查 API Key 配置。")

In [ ]:
# 检查模型加载后的 GPU 内存使用
print("模型加载后的 GPU 内存:")
print_gpu_memory()

## 下一步？

现在您已经了解了基本的 LLM 功能，您可以探索：

1. **训练您自己的小模型** - 参见 `03_training_demo.ipynb`
2. **微调预训练模型** - 参见 `04_fine_tuning_demo.ipynb`
3. **理解注意力机制** - 参见 `05_attention_visualization.ipynb`

每个 Notebook 都建立在之前学到的概念之上。